# theta_interpretation: entropy convergence vs. $\sigma$

Goal: sweep the SDE diffusion coefficient `sigma` for the **correctly-specified**
scalar Gaussian case (`terms=['x2']`) and reproduce a three-curve plot vs.
$\sigma^2$ (log x-axis):

- $H(p_*)$ -- the *exact* entropy of the true max-entropy distribution. We
  can compute this in closed form here (not just bound it) precisely
  *because* the potential set is correctly specified: for `phi(x) = x^2`
  and target `N(0, data_sigma^2)`, the max-entropy distribution matching
  `E[x^2] = data_sigma^2` **is** `N(0, data_sigma^2)` itself, whose entropy
  is `0.5 * log(2*pi*e*data_sigma^2)`. Constant in `sigma` (the SDE
  diffusion coefficient) -- it depends only on `data_sigma`.
- $H(p_1^\sigma)$ -- `codes/utils_entropy.py`'s `entropy_bound()` (MGD
  Prop. 4.3 / Eq. 30): a *lower bound* on the entropy of the generated
  distribution at `t=1`, built from the SDE integral `H(p_0) + int dH_t
  dt`. This is "the bound comes from MGD" quantity -- a theoretical
  guarantee, not a measurement of the empirical `xt` samples.
- $H^1$ -- a naive/plug-in histogram estimate of the differential entropy of
  the *actual generated samples* `xt` at `t=1`, using `codes/utils_entropy.py`'s
  `entropy()` (equal-count histogram binning + `-sum p log p`). Unlike the
  MGD bound, this one measures how well-mixed the realized finite-sample
  walkers actually are.

**Caveat on `H^1` naming**: this notebook's mapping of the pasted figure's
three curves to these three specific functions is my best-supported
reconstruction from what's actually implemented in this codebase (see the
chat explanation) -- I could not independently verify it against whatever
paper/source produced the original image, since no earlier notebook in this
repo reproduces that exact figure. If `H^1` was meant to be something else
(e.g. a different-order bound rather than an empirical estimate), the
labels/curves below would need to change accordingly.

**Expected qualitative behaviour**: $H(p_1^\sigma)$ (the theoretical bound)
should stay close to $H(p_*)$ across the whole `sigma` range, since it's a
provable guarantee independent of whether the finite walker population has
actually mixed well. $H^1$ (the naive estimate on the realized samples)
should need a *larger* `sigma` to converge up to $H(p_*)$, since more
diffusion means more stochastic exploration in the same `nt` integration
steps, i.e. a better-mixed, more accurately Gaussian-looking empirical
sample at small/moderate `sigma`.

This notebook imports and reuses `run_SDE.py`'s `make_args()`/
`run_and_diagnose()`, exactly like `theta_interpretation.ipynb`, so it never
drifts from the CLI script's run/save/naming logic.


In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

root = Path.cwd()
if not (root / 'run_SDE.py').exists():
    root = root / 'theta_interpretation'  # allow running the notebook from the repo root
sys.path.insert(0, str(root))
sys.path.insert(0, str(root.parent / 'codes'))

from run_SDE import make_args, run_and_diagnose, get_scalar_potentials, device
from utils_entropy import entropy_bound, entropy, standard_gaussian_entropy  # codes/utils_entropy.py

print('device:', device)


device: cpu


## Parameters

`SIGMA_LIST` is the swept quantity (SDE diffusion coefficient). Everything
else is a fixed override threaded through `make_args(['x2'], sigma=..., **PARAM_OVERRIDES)`
-- see `theta_interpretation.ipynb`'s parameter-override cell for why this
is necessary (`make_args()`'s un-overridden defaults mirror the CLI
defaults, not anything you'd type in this cell).


In [2]:
SIGMA_LIST = np.geomspace(0.3, 6.0, 10)   # sigma^2 spans roughly [0.1, 36], log-spaced

N1 = 3000
DATA_SIGMA = 1.0
NT = 800
SCHEDULE_EXPONENT = 2
INTERPOLANT = 'Cos'
REGULARIZATION = 1e-1
LAM = 5e-6
N_SUBSAMPLE = 100
BATCH_SIZE = None
N_BINS = 100                 # histogram bins for both run_SDE.py's own diagnostics AND the H^1 estimator below
MOMENT_THRESHOLD = 1e-8
SEED = 0
FORCE_RERUN = False
NO_SAVE_AUX_MOMENTS = False

PARAM_OVERRIDES = dict(
    n1=N1, data_sigma=DATA_SIGMA, nt=NT,
    schedule_exponent=SCHEDULE_EXPONENT, interpolant=INTERPOLANT,
    regularization=REGULARIZATION, lam=LAM, n_subsample=N_SUBSAMPLE,
    batch_size=BATCH_SIZE, n_bins=N_BINS, moment_threshold=MOMENT_THRESHOLD,
    seed=SEED, force_rerun=FORCE_RERUN, no_save_aux_moments=NO_SAVE_AUX_MOMENTS,
    label='entropy_convergence',
)


## Sweep `sigma`, run the experiment for each value

Each `sigma` gets its own config folder (`build_config_name()` hashes
`sigma` into the name), so re-running this cell after the first time just
reloads the cached results instead of re-fitting.


In [3]:
runs = {}
for sigma_val in SIGMA_LIST:
    args = make_args(['x2'], sigma=float(sigma_val), outdir=str(root), **PARAM_OVERRIDES)
    runs[float(sigma_val)] = run_and_diagnose(args)
    print(f"sigma={sigma_val:.4f} done: {runs[float(sigma_val)]['config']}")


2026-09-03 15:09:35  INFO      Config: thetainterp_sigmadata1.0_sigma0.3_nt800_n1_3000_lam5e-06_seed_0_terms8e683187_entropy_convergence


2026-09-03 15:09:35  INFO      Experiment folder: /home/chiaraz/phd/MGD-for-Maximum-Entropy-Generation/theta_interpretation/experiments/thetainterp_sigmadata1.0_sigma0.3_nt800_n1_3000_lam5e-06_terms8e683187_entropy_convergence/thetainterp_sigmadata1.0_sigma0.3_nt800_n1_3000_lam5e-06_seed_0_terms8e683187_entropy_convergence


2026-09-03 15:09:35  INFO      Arguments: {'timestamp': None, 'n1': 3000, 'data_sigma': 1.0, 'terms': ['x2'], 'nt': 800, 'sigma': 0.3, 'schedule_exponent': 2, 'interpolant': 'Cos', 'regularization': 0.1, 'lam': 5e-06, 'n_subsample': 100, 'batch_size': None, 'n_bins': 100, 'moment_threshold': 1e-08, 'outdir': '/home/chiaraz/phd/MGD-for-Maximum-Entropy-Generation/theta_interpretation', 'label': 'entropy_convergence', 'force_rerun': False, 'no_save_aux_moments': False, 'seed': 0}


2026-09-03 15:09:35  INFO      Running experiment: thetainterp_sigmadata1.0_sigma0.3_nt800_n1_3000_lam5e-06_seed_0_terms8e683187_entropy_convergence


2026-09-03 15:09:36  INFO      terms: ['x2']


...

After _solve_regularised: CPU=0.73 GB
Stacking outputs
Everything stacked: CPU=0.73 GB
Returning
2026-09-03 15:12:27  INFO      SDE integration finished in 4.1 s


2026-09-03 15:12:29  INFO      Final theta (order matches ['x2']): [-0.8471596837043762]


2026-09-03 15:12:29  INFO      Reference (unverified against this run): for target N(0, data_sigma^2) under p(x) ~ exp(sum_i theta_i * phi_i(x)), theta on x^2 is expected near -0.500000. Fitted theta_x2[-1] = -0.847160


2026-09-03 15:12:29  INFO      Saved diagnostic figures to /home/chiaraz/phd/MGD-for-Maximum-Entropy-Generation/theta_interpretation/experiments/thetainterp_sigmadata1.0_sigma6.0_nt800_n1_3000_lam5e-06_terms8e683187_entropy_convergence/thetainterp_sigmadata1.0_sigma6.0_nt800_n1_3000_lam5e-06_seed_0_terms8e683187_entropy_convergence/figures


sigma=6.0000 done: thetainterp_sigmadata1.0_sigma6.0_nt800_n1_3000_lam5e-06_seed_0_terms8e683187_entropy_convergence


## Compute the three entropy quantities

- `H_star`: exact, computed once via `standard_gaussian_entropy(1, log_det_cov=2*log(data_sigma))`
  -- `standard_gaussian_entropy(d, log_det_cov)` is `codes/utils_entropy.py`'s
  plain closed-form Gaussian entropy; `log_det_cov = log(data_sigma**2)` for
  our scalar `N(0, data_sigma^2)` target.
- `H_bound_list`: `entropy_bound(...)['H_bound']` per sigma -- the MGD
  Prop. 4.3 lower bound (`check_p0=False` here purely to skip that extra
  diagnostic and keep the sweep fast; it doesn't affect `H_bound` itself).
- `H1_list`: `entropy(xt, n_bins=N_BINS)` per sigma, applied to the
  generated walker samples `xt` at `t=1` (NOT the raw data `x1`, which
  wouldn't depend on `sigma` at all).


In [4]:
H_star = float(standard_gaussian_entropy(1, log_det_cov=2 * np.log(DATA_SIGMA)))
print('H(p_*) =', H_star)

sigmas_sorted = sorted(runs.keys())
H_bound_list, H1_list = [], []

for sigma_val in sigmas_sorted:
    run = runs[sigma_val]
    potentials = get_scalar_potentials(run['args'].terms)

    eb = entropy_bound({'run': run['result']}, 'run', potentials, device=device, check_p0=False)
    H_bound_list.append(eb['H_bound'])

    xt_np = run['result']['xt'].detach().cpu().numpy().ravel()
    H1_list.append(entropy(xt_np, n_bins=N_BINS))

    print(f"sigma={sigma_val:.4f}  H_bound={eb['H_bound']:.4f}  H^1={H1_list[-1]:.4f}")

sigmas_sorted = np.array(sigmas_sorted)
H_bound_arr = np.array(H_bound_list)
H1_arr = np.array(H1_list)


H(p_*) = 1.4189385332046727
sigma=0.3000  H_bound=1.4391  H^1=1.4063
sigma=0.4185  H_bound=1.4405  H^1=1.4071
sigma=0.5838  H_bound=1.4412  H^1=1.4055
sigma=0.8143  H_bound=1.4416  H^1=1.4076
sigma=1.1359  H_bound=1.4418  H^1=1.4153
sigma=1.5846  H_bound=1.4420  H^1=1.4141
sigma=2.2104  H_bound=1.4420  H^1=1.4132
sigma=3.0834  H_bound=1.4418  H^1=1.4143
sigma=4.3012  H_bound=1.4414  H^1=1.4111
sigma=6.0000  H_bound=1.4406  H^1=1.4041


## Moment matching per sigma (convergence check)

Same diagnostic `theta_interpretation.ipynb` shows for a single run --
`plot_moment_matching()` (`codes/check_moments.py`), already computed and
saved to each run's own `fig_dir` by `run_and_diagnose()` (`save_diagnostics()`
in `run_SDE.py`) -- displayed here for every `sigma` in the sweep so you can
see whether the fit actually converged at each point, not just trust the
final `H_bound`/`H^1` numbers. If the relative moment-matching error hasn't
dropped below `moment_threshold` for a given `sigma`, that run's entropy
values above aren't trustworthy.


In [ ]:
for sigma_val in sigmas_sorted:
    run = runs[float(sigma_val)]
    print(f"--- sigma = {sigma_val:.4f}  (sigma^2 = {sigma_val**2:.4f}) ---")
    display(Image(filename=str(run['fig_dir'] / 'moment_matching.png')))


## Reproduce the figure


In [5]:
fig, ax = plt.subplots(figsize=(5, 4))

ax.axhline(H_star, color='red', lw=1.5, label=r'$H(p_*)$')
ax.plot(sigmas_sorted**2, H_bound_arr, 'o-', color='tab:blue', ms=5,
        label=r'$H(p_1^\sigma)$')
ax.plot(sigmas_sorted**2, H1_arr, 'o--', color='black', ms=5,
        label=r'$H^1$')

ax.set_xscale('log')
ax.set_xlabel(r'$\sigma^2$')
ax.set_ylabel('entropy')
ax.legend(frameon=False)
fig.tight_layout()

out_dir = root / 'entropy_convergence'
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / 'entropy_convergence.png', dpi=150, bbox_inches='tight')
plt.show()


/tmp/ipykernel_67763/1075097991.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
